In [ ]:
# Parameters
slice_name = "mfportal_presubmit_test"


#### Measurement Framework Library
# MFPortal API Test: Add Meas Network Before Submit

Tests `MFPortal.add_meas_network_presubmit()` from `mflib/mfportal.py`.

`MFPortal.add_meas_network()` retrofits a FABNetv6 meas network onto an
*already-submitted* slice via `modify`. In practice that modify has been
unreliable: the new network's reservation is created with no error at
the time, but the network never shows up in the topology afterward and
its interfaces later show up in `Closed` state -- see
[fabnetv6-modify-network-not-in-topology.md](../mflib/notes/fabnetv6-modify-network-not-in-topology.md)
for the full writeup.

`add_meas_network_presubmit()` is the workaround under test: wire the
FABNetv6 NIC(s) onto the local, *not-yet-submitted* topology, so the
network is requested as part of the slice's first `submit()` (a
`create`) instead of a later `modify`. This notebook builds a plain
3-node slice, adds the meas network before the first submit, submits,
and then checks `list_interfaces()` to see whether the interfaces come
up `Active` with a real subnet/IP this time, instead of `Closed`.

## General Imports

In [ ]:
import os
import json
import traceback

## Import MFPortal

This imports `MFPortal` from `mflib.mfportal`. If you have trouble
importing `mflib`, see [Install MFLib](./mflib_install.ipynb).

In [ ]:
import mflib
print(f"MFLib version  {mflib.__version__} ")

from mflib.mfportal import MFPortal

## Setup Experiment Slice

### Import fablib

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

try:
    fablib = fablib_manager()
    fablib.show_config()
except Exception as e:
    print(f"Exception: {e}")

### Set Slice Information

In [ ]:
%%time

# Sites without PTP + NCSA where meas node is located + problem sites
[site1, site2, site3] = fablib.get_random_sites(
    count=3,
    avoid=["DALL","GPN","LBNL","RENC","SALT","TACC","UKY","WASH","NCSA",
           "LOSA","GATECH","INDI","MAX","MASS","NEWY","SRI","UCSD"],
)

node1_name = 'Node1'
node2_name = 'Node2'
node3_name = 'Node3'

print(f"Setting up slice {slice_name}")
print(f"Using sites {site1}, {site2}, {site3}")

### Create Experiment Topology

Plain nodes only, no extra dataplane NICs/networks -- `add_meas_network_presubmit()`
adds its own FABNetv6 NIC per node via `node.add_fabnet()`, so nothing else is needed here.

In [ ]:
try:
    # Create Slice
    slice = fablib.new_slice(name=slice_name)

    node1 = slice.add_node(name=node1_name, site=site1)
    node2 = slice.add_node(name=node2_name, site=site2)
    node3 = slice.add_node(name=node3_name, site=site3)

    print(f"Slice Topology Done.")
except Exception as e:
    print(f"Exception: {e}")

### Add the meas network -- BEFORE the first submit

This is the method under test. `slice` has no `slice_id` yet (it hasn't
been submitted), so this wires the FABNetv6 NIC(s) onto the local
topology only -- nothing is sent to the orchestrator until the
`slice.submit()` call in the next cell. Compare this against
`MFPortal.add_meas_network(slice)`, which is for a slice that's already
been submitted and drives its own `submit()`/`wait()` internally.

In [ ]:
wired = MFPortal.add_meas_network_presubmit(slice)
print(f"Wired FABNetv6 NIC(s) for: {wired}")

### Submit the Slice

A normal initial slice submission -- the meas network(s) go through the
orchestrator's `create` path here, not `modify`.

In [ ]:
%%time
try:
    # Submit Slice Request
    print(f'Submitting the new slice, "{slice_name}"...')
    slice.submit()
    print(f'{slice_name} creation done.')

except Exception as e:
    print(f"Slice Fail: {e}")
    traceback.print_exc()

### Verify the meas network

`list_interfaces()` after a fresh `update()` -- this is the check that
matters. Look for the `meas-net6_IPv6_<site>_nic-p1` interfaces: if this
workaround holds, they should show up `Active` with a real IP, not
`Closed` the way the modify path did.

In [ ]:
slice.update()
slice.list_interfaces(refresh=True)

### Verify the meas network's subnet/gateway/IP

The real pass/fail signal for the underlying bug: does the network
actually have a subnet and gateway now? `get_meas_net()` covers every
node with a wired-up FABNetv6 NIC and calls `assign_static_fabnet6_ip()`
for each -- if the presubmit path worked, this should return real
`node_ipv6`/`meas_net_subnet`/`gw_v6` values instead of failing or
coming back empty.

In [ ]:
MFPortal.get_meas_net(slice)

-----
# Slice Setup Is Complete

If `list_interfaces()` above shows the meas-net6 interfaces `Active`
with real IPs, and `get_meas_net()` returned real subnet/gateway/IP
data, that confirms the presubmit (initial-creation) path avoids the
modify-path bug in
[fabnetv6-modify-network-not-in-topology.md](../mflib/notes/fabnetv6-modify-network-not-in-topology.md).
If the interfaces still show `Closed` or the network still can't be
found, the bug isn't specific to `modify` after all, and that note
needs to be revised.

-----